In [0]:
from pyspark.sql.functions import *
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, StructType, StructField, ArrayType, DoubleType, IntegerType
import pandas as pd
from pyarrow import *
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression  
from pyspark.ml.feature import *
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pyarrow.parquet as pq


In [0]:
lreg_df = spark.read.table("hive_metastore.default.lreg_df")

In [0]:
final_df = lreg_df.drop("AO","NOME","TIPOINST","TAGCOM","REDE","ID_prefix","ID_OBJECTO")

In [0]:
final_df = final_df.withColumn("AMPM_flag", when(col("AM_PM")=="PM", 1.0).otherwise(0.0))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window as W

def make_k_ahead_labels(df, id_col="ID", time_col="DATE",
                        label_col="has_falha", k=24):
    """
    Replaces `label_col` with its k-steps-ahead version within each ID,
    and drops rows that don't have a future label (tail).
    """
    w = W.partitionBy(id_col).orderBy(time_col)
    df_k = df.withColumn(label_col, F.lead(F.col(label_col), k).over(w))
    return df_k.filter(F.col(label_col).isNotNull())

# Example usage
k = 24  # 6h if your step is 15 minutes
df_k = make_k_ahead_labels(final_df, id_col="ID", time_col="DATE",
                           label_col="has_falha", k=k)

In [0]:
display(df_k)

In [0]:
# Fixed column sets
WEATHER_COLS = ["temperature", "precipitation", "wind_speed", "humidity"]

BASE_NUMERIC = [
    "INTENSITY","TENSION","H_LIM_I","H_LIM_T",
    "MAVERAGE_2H_I","MAVERAGE_2H_T","MAVERAGE_1D_I","MAVERAGE_1D_T",
    "EVENT_COUNT_I","EVENT_COUNT_T",
    "TIME_OVER_LIMIT_I","TIME_OVER_LIMIT_T",
    "DAY_OF_WEEK","DAY_OF_MONTH","DAY_OF_YEAR","HOUR_OF_DAY",
    "AMPM_flag"  # 0/1; we scale it together with the rest to keep the vector layout identical
]

def make_pipeline(use_weather: bool,
                  regParam=0.0, elasticNetParam=0.0, maxIter=100,
                  id_col="ID", conc_col="CONCELHO", label_col="has_falha"):
    # 1) categorical → index → OHE (dropLast=True for a canonical basis)
    idx_id   = StringIndexer(inputCol=id_col,   outputCol="ID_idx",        handleInvalid="keep")
    idx_conc = StringIndexer(inputCol=conc_col, outputCol="CONCELHO_idx",  handleInvalid="keep")
    ohe = OneHotEncoder(
        inputCols=["ID_idx","CONCELHO_idx"],
        outputCols=["ID_ohe","CONCELHO_ohe"],
        dropLast=True
    )

    # 2) numeric → assembler → scaler (stable names)
    numeric_cols = BASE_NUMERIC + (WEATHER_COLS if use_weather else [])
    num_asm  = VectorAssembler(inputCols=numeric_cols, outputCol="num_features")
    scaler   = StandardScaler(inputCol="num_features", outputCol="num_scaled",
                              withMean=True, withStd=True)

    # 3) final features (freeze order!)
    feats = VectorAssembler(inputCols=["ID_ohe","CONCELHO_ohe","num_scaled"], outputCol="features")

    # 4) LR
    lr = LogisticRegression(
        featuresCol="features", labelCol=label_col,
        regParam=regParam, elasticNetParam=elasticNetParam,
        maxIter=maxIter, standardization=False
    )
    # (standardization=False because we already scaled numerics; OHE parts don’t need scaling)

    return Pipeline(stages=[idx_id, idx_conc, ohe, num_asm, scaler, feats, lr]), numeric_cols


In [0]:
cutoff_date = "2023-11-30"
train = df_k.filter(col("DATE") < cutoff_date)
test = df_k.filter(col("DATE") >= cutoff_date)

In [0]:
# Example: train both versions on the *same* train_df
regParam = 0.0
elasticNetParam = 0.0
seed = 42

pipe_w, num_w = make_pipeline(True,  regParam, elasticNetParam)
pipe_n, num_n = make_pipeline(False, regParam, elasticNetParam)

model_w = pipe_w.fit(train)
model_n = pipe_n.fit(train)


In [0]:
predictions_w = model_w.transform(test)

display(predictions_w.select("has_falha", "prediction", "probability"))

In [0]:
predictions_n = model_n.transform(test)

display(predictions_n.select("has_falha", "prediction", "probability"))

In [0]:
evaluator_w = BinaryClassificationEvaluator(labelCol="has_falha", rawPredictionCol="rawPrediction")
print("AUC:", evaluator_w.evaluate(predictions_w))


In [0]:
evaluator_n = BinaryClassificationEvaluator(labelCol="has_falha", rawPredictionCol="rawPrediction")
print("AUC:", evaluator_n.evaluate(predictions_n))


In [0]:
confusion_df_w = (
    predictions_w.groupBy("has_falha", "prediction")
    .count()
    .orderBy("has_falha", "prediction")
)
display(confusion_df_w)

In [0]:
confusion_df_n = (
    predictions_n.groupBy("has_falha", "prediction")
    .count()
    .orderBy("has_falha", "prediction")
)
display(confusion_df_n)

In [0]:
# === Uniform, DBFS-safe evaluator (adds filename tag like "_weather") ===
import os, json, time, math, datetime as dt
import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pyspark.sql import functions as F, Window as W
from pyspark.sql.types import *
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

def _counts_to_metrics(tp, fp, tn, fn):
    tot  = tp + fp + tn + fn
    acc  = (tp + tn) / tot if tot else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    bal  = 0.5 * (rec + spec)
    f1   = (2 * prec * rec) / (prec + rec) if (prec + rec) else 0.0
    den  = math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    mcc  = ((tp*tn - fp*fn) / den) if den else 0.0
    return acc, prec, rec, spec, bal, f1, mcc

def _plot_cm(cm, title, path):
    fig, ax = plt.subplots(figsize=(4,4), dpi=160)
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title); fig.colorbar(im)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Pred 0","Pred 1"])
    ax.set_yticklabels(["True 0","True 1"])
    thr = cm.max()/2 if cm.size else 0
    for (i,j), v in np.ndenumerate(cm):
        ax.text(j, i, f"{int(v):,}", ha="center", va="center",
                color="white" if v > thr else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    fig.tight_layout(); os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, bbox_inches="tight"); plt.close(fig)

def evaluate_predictions_spark(predictions,
                               labelCol="has_falha",
                               probCol="probability",
                               rawCol="rawPrediction",
                               model_name="logreg_v1",
                               report_base="dbfs:/reports",
                               delta_log_path="dbfs:/reports/metrics_delta",
                               file_tag="_weather"):   # <<--- NEW
    # normalize tag ("" or startswith "_")
    file_tag = "" if not file_tag else (file_tag if file_tag.startswith("_") else f"_{file_tag}")

    dbutils.fs.mkdirs(report_base)
    dbutils.fs.mkdirs(delta_log_path)

    # 1) Extract p(class=1)
    df = (predictions
          .withColumn("_prob_arr", vector_to_array(F.col(probCol)))
          .withColumn("proba", F.col("_prob_arr")[1].cast(DoubleType()))
          .withColumn("label_int", F.col(labelCol).cast(IntegerType()))
          .select("label_int", "proba", rawCol))
    df = df.filter(F.col("proba").isNotNull() & F.col("label_int").isNotNull())

    # Totals
    P, N = df.select(
        F.sum(F.when(F.col("label_int")==1, 1).otherwise(0)),
        F.sum(F.when(F.col("label_int")==0, 1).otherwise(0))
    ).first()
    P, N = int(P), int(N)

    # 2) AUC / AUPRC on original predictions
    auc  = float(BinaryClassificationEvaluator(labelCol=labelCol, rawPredictionCol=rawCol, metricName="areaUnderROC").evaluate(predictions))
    aupr = float(BinaryClassificationEvaluator(labelCol=labelCol, rawPredictionCol=rawCol, metricName="areaUnderPR").evaluate(predictions))

    # 3) Confusion @ 0.5
    df05 = df.withColumn("pred05", (F.col("proba") >= 0.5).cast("int"))
    tp05, fp05, fn05, tn05 = df05.select(
        F.sum(F.when((F.col("pred05")==1) & (F.col("label_int")==1), 1).otherwise(0)),
        F.sum(F.when((F.col("pred05")==1) & (F.col("label_int")==0), 1).otherwise(0)),
        F.sum(F.when((F.col("pred05")==0) & (F.col("label_int")==1), 1).otherwise(0)),
        F.sum(F.when((F.col("pred05")==0) & (F.col("label_int")==0), 1).otherwise(0))
    ).first()
    tp05, fp05, fn05, tn05 = map(int, (tp05, fp05, fn05, tn05))
    acc05, prec05, rec05, spec05, bal05, f105, mcc05 = _counts_to_metrics(tp05, fp05, tn05, fn05)

    # 4) Best-F1 threshold via bucketing
    dfb = df.withColumn("bucket", (F.floor(F.col("proba") * 1000) / 1000.0).cast(DoubleType()))
    byb = (dfb.groupBy("bucket")
              .agg(F.sum(F.when(F.col("label_int")==1, 1).otherwise(0)).alias("pos"),
                   F.sum(F.when(F.col("label_int")==0, 1).otherwise(0)).alias("neg")))
    w = W.orderBy(F.col("bucket").desc())
    byb = (byb
           .withColumn("cum_pos", F.sum("pos").over(w))
           .withColumn("cum_neg", F.sum("neg").over(w))
           .withColumn("tp", F.col("cum_pos"))
           .withColumn("fp", F.col("cum_neg"))
           .withColumn("fn", F.lit(P) - F.col("cum_pos"))
           .withColumn("tn", F.lit(N) - F.col("cum_neg"))
           .withColumn("prec", F.when(F.col("tp")+F.col("fp")>0, F.col("tp")/(F.col("tp")+F.col("fp"))).otherwise(F.lit(0.0)))
           .withColumn("rec",  F.when(F.col("tp")+F.col("fn")>0, F.col("tp")/(F.col("tp")+F.col("fn"))).otherwise(F.lit(0.0)))
           .withColumn("f1",   F.when(F.col("prec")+F.col("rec")>0, 2*F.col("prec")*F.col("rec")/(F.col("prec")+F.col("rec"))).otherwise(F.lit(0.0))))
    best = byb.orderBy(F.col("f1").desc(), F.col("bucket").desc()).limit(1).first()
    best_thr = float(best["bucket"])
    tpB, fpB, fnB, tnB = map(int, (best["tp"], best["fp"], best["fn"], best["tn"]))
    accB, precB, recB, specB, balB, f1B, mccB = _counts_to_metrics(tpB, fpB, tnB, fnB)

    # 5) Save plots/JSON to /tmp then copy to DBFS (filenames tagged)
    ts = time.strftime("%Y%m%d_%H%M%S")
    local_dir = f"/tmp/{model_name}_{ts}"
    os.makedirs(local_dir, exist_ok=True)

    cm05   = np.array([[tn05, fp05],[fn05, tp05]])
    cmbest = np.array([[tnB,  fpB ],[fnB,  tpB ]])

    cm05_name   = f"cm_thr_0.50{file_tag}.png"
    cmbest_name = f"cm_thr_{best_thr:.3f}{file_tag}.png"
    report_name = f"report{file_tag}.json"

    cm05_local   = os.path.join(local_dir, cm05_name)
    cmbest_local = os.path.join(local_dir, cmbest_name)
    _plot_cm(cm05,   f"{model_name} | thr=0.50", cm05_local)
    _plot_cm(cmbest, f"{model_name} | thr={best_thr:.3f}", cmbest_local)

    db_dir = f"{report_base}/{model_name}/{ts}"
    dbutils.fs.mkdirs(db_dir)
    dbutils.fs.cp(f"file:{cm05_local}",   f"{db_dir}/{cm05_name}",   True)
    dbutils.fs.cp(f"file:{cmbest_local}", f"{db_dir}/{cmbest_name}", True)

    summary = {
        "model_name": model_name,
        "timestamp": ts,
        "n_samples": int(P+N),
        "pos_rate": float(P/(P+N)) if (P+N) else 0.0,
        "auc": auc, "ap": aupr,
        "thr_fixed": 0.5,
        "metrics_fixed": {"tp":tp05,"fp":fp05,"tn":tn05,"fn":fn05,
                          "acc":acc05,"prec":prec05,"rec":rec05,"f1":f105,"spec":spec05,"bal_acc":bal05,"mcc":mcc05},
        "thr_bestF1": best_thr,
        "metrics_bestF1": {"tp":tpB,"fp":fpB,"tn":tnB,"fn":fnB,
                           "acc":accB,"prec":precB,"rec":recB,"f1":f1B,"spec":specB,"bal_acc":balB,"mcc":mccB},
        "confusion_fixed_path": f"{db_dir}/{cm05_name}",
        "confusion_best_path":  f"{db_dir}/{cmbest_name}"
    }
    json_local = os.path.join(local_dir, report_name)
    with open(json_local, "w") as f:
        json.dump(summary, f, indent=2)
    dbutils.fs.cp(f"file:{json_local}", f"{db_dir}/{report_name}", True)

    # 6) Append metrics to a Delta table (unchanged)
    schema = StructType([
        StructField("model", StringType()), StructField("ts_utc", StringType()),
        StructField("threshold", DoubleType()),
        StructField("auc", DoubleType()), StructField("auprc", DoubleType()),
        StructField("accuracy", DoubleType()), StructField("f1", DoubleType()),
        StructField("precision", DoubleType()), StructField("recall", DoubleType()),
        StructField("specificity", DoubleType()), StructField("balanced_accuracy", DoubleType()),
        StructField("mcc", DoubleType()),
        StructField("tp", LongType()), StructField("fp", LongType()),
        StructField("tn", LongType()), StructField("fn", LongType()),
        StructField("n_samples", LongType()),
    ])
    now = dt.datetime.utcnow().isoformat()
    rows = [
        (model_name, now, 0.5,      auc, aupr, acc05, f105, prec05, rec05, spec05, bal05, mcc05, tp05, fp05, tn05, fn05, int(P+N)),
        (model_name, now, best_thr, auc, aupr, accB, f1B,  precB,  recB,  specB,  balB,  mccB,  tpB,  fpB,  tnB,  fnB,  int(P+N)),
    ]
    spark.createDataFrame(rows, schema)\
         .write.format("delta").mode("append").save(delta_log_path)

    print(f"=== {model_name} | {ts} ===")
    print(f"N={int(P+N)}, Pos rate={P/(P+N):.4f}")
    print(f"AUC={auc:.4f}, AUPRC={aupr:.4f}")
    print(f"[thr=0.50]   ACC={acc05:.4f}  PREC={prec05:.4f}  REC={rec05:.4f}  F1={f105:.4f}")
    print(f"[thr={best_thr:.3f}] ACC={accB:.4f}  PREC={precB:.4f}  REC={recB:.4f}  F1={f1B:.4f}")
    print("Saved:", f"{db_dir}/{cm05_name}")
    print("Saved:", f"{db_dir}/{cmbest_name}")
    print("Report:", f"{db_dir}/{report_name}")
    print("Logged metrics to Delta:", delta_log_path)

    return {
        "report_dir": db_dir,
        "best_thr": best_thr,
        "auc": auc, "auprc": aupr,
        "fixed": {"acc":acc05,"prec":prec05,"rec":rec05,"f1":f105},
        "bestF1":{"acc":accB,"prec":precB,"rec":recB,"f1":f1B},
        "cm_fixed_path": f"{db_dir}/{cm05_name}",
        "cm_best_path":  f"{db_dir}/{cmbest_name}",
        "report_path":   f"{db_dir}/{report_name}"
    }


In [0]:
# predictions = lr_model.transform(test_df)
res = evaluate_predictions_spark(
    predictions_w,
    labelCol="has_falha",
    probCol="probability",
    rawCol="rawPrediction",
    model_name="logreg_v1",
    file_tag="_weather"        # <<— adds suffix to PNGs + JSON
)
from PIL import Image
from IPython.display import display

display(Image.open(res["cm_fixed_path"].replace("dbfs:/", "/dbfs/")))
display(Image.open(res["cm_best_path"].replace("dbfs:/", "/dbfs/")))


In [0]:
# predictions = lr_model.transform(test_df)
res = evaluate_predictions_spark(
    predictions_n,
    labelCol="has_falha",
    probCol="probability",
    rawCol="rawPrediction",
    model_name="logreg_v1",
    file_tag="_normal"        # <<— adds suffix to PNGs + JSON
)
from PIL import Image
from IPython.display import display

display(Image.open(res["cm_fixed_path"].replace("dbfs:/", "/dbfs/")))
display(Image.open(res["cm_best_path"].replace("dbfs:/", "/dbfs/")))


In [0]:
# Cell 1 — imports + helpers
import os, time, math, builtins
import numpy as np
import pandas as pd

from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.feature import OneHotEncoderModel, StringIndexerModel, VectorAssembler, StandardScalerModel

def _find_lr_model(pipeline: PipelineModel) -> LogisticRegressionModel:
    """Return the fitted LogisticRegressionModel from a PipelineModel (search from the end)."""
    for st in reversed(pipeline.stages):
        if isinstance(st, LogisticRegressionModel):
            return st
    raise ValueError("No fitted LogisticRegressionModel found in PipelineModel.")

def _stage_by_output(pipeline: PipelineModel):
    """Map each stage's output column(s) → the stage object (handles OHE multi-output)."""
    by_out = {}
    for st in pipeline.stages:
        if isinstance(st, OneHotEncoderModel):
            for oc in st.getOutputCols():
                by_out[oc] = st
        elif hasattr(st, "getOutputCol"):
            try:
                oc = st.getOutputCol()
                if oc:
                    by_out[oc] = st
            except Exception:
                pass
    return by_out

def _indexer_by_output(pipeline: PipelineModel):
    """Map StringIndexer output columns (e.g., 'ID_idx') → their StringIndexerModel (for labels)."""
    d = {}
    for st in pipeline.stages:
        if isinstance(st, StringIndexerModel):
            d[st.getOutputCol()] = st
    return d


In [0]:
# Patch: robust feature-name builder that expands vector inputs and OHE
import builtins
import numpy as np
from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.feature import (
    StringIndexerModel, OneHotEncoderModel, VectorAssembler, StandardScalerModel
)

def _find_lr_and_features_col(pipeline_model: PipelineModel):
    lr = None
    for st in reversed(pipeline_model.stages):
        if isinstance(st, LogisticRegressionModel):
            lr = st
            break
    if lr is None:
        raise ValueError("Could not find LogisticRegressionModel in pipeline.")
    return lr, lr.getFeaturesCol()

def build_feature_names_and_groups(
    pipeline: PipelineModel,
    featuresCol: str | None = None,
    id_ohe_col: str = "ID_ohe",
    concelho_ohe_col: str = "CONCELHO_ohe",
):
    # 1) LR + its features column
    lr, lr_feats = _find_lr_and_features_col(pipeline)
    feats_col = featuresCol or lr_feats

    # 2) Find the final VectorAssembler that outputs feats_col
    final_va = None
    for st in pipeline.stages:
        if isinstance(st, VectorAssembler) and st.getOutputCol() == feats_col:
            final_va = st
            break
    if final_va is None:
        raise ValueError(f"Could not find VectorAssembler that outputs '{feats_col}'.")

    inputs = list(final_va.getInputCols())

    # 3) Build maps for vector dimensions:
    #    - vec_dims: outputCol -> number of inputs (dimension) for any VectorAssembler
    #    - scaler_in: scaler_out -> scaler_in (so we can resolve back to its upstream VA)
    vec_dims = {}
    scaler_in = {}
    for st in pipeline.stages:
        if isinstance(st, VectorAssembler):
            vec_dims[st.getOutputCol()] = len(st.getInputCols())
        elif isinstance(st, StandardScalerModel):
            scaler_in[st.getOutputCol()] = st.getInputCol()

    def dim_for(colname: str) -> int:
        # direct VA output?
        if colname in vec_dims:
            return int(vec_dims[colname])
        # StandardScaler output -> resolve dimension via its input VA
        if colname in scaler_in:
            upstream = scaler_in[colname]
            return int(vec_dims.get(upstream, 1))
        # otherwise it's a scalar column
        return 1

    # 4) StringIndexer labels for OHE readability (optional; we still work if missing)
    id_labels, conc_labels = None, None
    for st in pipeline.stages:
        if isinstance(st, StringIndexerModel):
            if st.getOutputCol() == "ID_idx":
                id_labels = list(st.labels)
            elif st.getOutputCol() == "CONCELHO_idx":
                conc_labels = list(st.labels)

    # 5) OHE sizes & dropLast
    ohe = None
    for st in pipeline.stages:
        if isinstance(st, OneHotEncoderModel):
            ohe = st
            break
    if ohe is None:
        raise ValueError("Could not find OneHotEncoderModel in pipeline.")

    out_cols = ohe.getOutputCols() if hasattr(ohe, "getOutputCols") else [ohe.getOutputCol()]
    sizes    = list(ohe.categorySizes) if hasattr(ohe, "categorySizes") else [ohe.getCategorySizes()]
    dropLast = bool(ohe.getDropLast())

    def ohe_dim_for(colname: str) -> int:
        if colname in out_cols:
            i = out_cols.index(colname)
            base = int(sizes[i])
        else:
            base = int(sizes[0])  # single-col fallback
        return base - (1 if dropLast else 0)

    id_expected       = ohe_dim_for(id_ohe_col) if id_ohe_col in inputs else 0
    concelho_expected = ohe_dim_for(concelho_ohe_col) if concelho_ohe_col in inputs else 0

    # 6) Build names in the exact order of final assembler inputs
    feature_names, groups = [], []

    def expand_ohe(out_col: str, friendly: str, labels, expected: int):
        nonlocal feature_names, groups
        use_n = builtins.max(0, builtins.min(len(labels) if labels else 0, expected))
        if labels:
            feature_names.extend([f"{friendly}=={lab}" for lab in labels[:use_n]])
        # pad any remaining (unseen bucket / capacity) to keep alignment with coefficients
        for k in range(expected - use_n):
            feature_names.append(f"{friendly}__extra[{k}]")
        groups.extend([friendly] * builtins.max(expected, 0))

    for c in inputs:
        if c == id_ohe_col:
            expand_ohe(c, "ID", id_labels, id_expected)
        elif c == concelho_ohe_col:
            expand_ohe(c, "CONCELHO", conc_labels, concelho_expected)
        else:
            d = dim_for(c)
            if d == 1:
                feature_names.append(c); groups.append(c)
            else:
                # expand vector inputs to per-dimension names
                feature_names.extend([f"{c}[{i}]" for i in range(d)])
                groups.extend([c] * d)

    return feature_names, groups


In [0]:
# Cell 3 — grouped feature importance for LogisticRegression + OHE pipeline

import builtins as b
import os, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyspark.ml import PipelineModel

def grouped_feature_importance_ohe(
    pipeline_model: PipelineModel,
    model_name: str = "logreg_ohe",
    out_base_dbfs: str = "dbfs:/reports_ohe",
    featuresCol: str | None = None,
    id_ohe_col: str = "ID_ohe",
    concelho_ohe_col: str = "CONCELHO_ohe",
    sort_by: str = "l2"   # one of: "l2", "l1", "rms_coef", "mean_abs"
):
    # 1) names & groups aligned to LR coefficients
    feature_names, groups = build_feature_names_and_groups(
        pipeline_model,
        featuresCol=featuresCol,
        id_ohe_col=id_ohe_col,
        concelho_ohe_col=concelho_ohe_col
    )

    # 2) get LR coefficients
    lr, _ = _find_lr_and_features_col(pipeline_model)
    coefs = np.array(lr.coefficients.toArray())
    if len(feature_names) != len(coefs):
        raise ValueError(f"Name/coef length mismatch after padding: {len(feature_names)} vs {len(coefs)}")

    # 3) feature-level table
    all_df = pd.DataFrame({
        "feature": feature_names,
        "group":   groups,
        "coef":    coefs,
        "abs_coef": np.abs(coefs),
    })

    # 4) group aggregation
    def agg_l2(x):  return float(np.sqrt(np.sum(np.square(x))))
    def agg_l1(x):  return float(np.sum(np.abs(x)))
    grouped = (all_df
               .groupby("group", as_index=False)
               .agg(n_dims=("coef","size"),
                    l2=("coef", agg_l2),
                    l1=("coef", agg_l1))
              )
    grouped["rms_coef"] = grouped["l2"] / np.sqrt(grouped["n_dims"].clip(lower=1))
    grouped["mean_abs"] = grouped["l1"] / grouped["n_dims"].clip(lower=1)

    # 5) save CSVs + quick bar plot (by group)
    ts = time.strftime("%Y%m%d_%H%M%S")
    out_dir_dbfs = f"{out_base_dbfs}/{model_name}/{ts}"
    out_dir_drv  = "/dbfs" + out_dir_dbfs[len("dbfs:"):]   # driver-visible path
    os.makedirs(out_dir_drv, exist_ok=True)

    all_csv      = os.path.join(out_dir_drv, "importance_all.csv")
    grouped_csv  = os.path.join(out_dir_drv, "importance_grouped.csv")
    all_df.to_csv(all_csv, index=False)
    grouped.to_csv(grouped_csv, index=False)

    # plot top groups by chosen metric
    order = grouped.sort_values(sort_by, ascending=False)
    top = order.head(30).iloc[::-1]  # reverse for barh
    plt.figure(figsize=(10, b.max(4.0, 0.30*len(top))))
    plt.barh(top["group"], top[sort_by])
    plt.title(f"{model_name} — group importance by {sort_by.upper()}")
    plt.xlabel(sort_by)
    plt.tight_layout()
    png_path = os.path.join(out_dir_drv, f"importance_groups_{sort_by}.png")
    plt.savefig(png_path, dpi=160, bbox_inches="tight"); plt.close()

    print("Saved:", all_csv)
    print("Saved:", grouped_csv)
    print("Saved:", png_path)

    return {
        "all_df": all_df,
        "grouped_df": grouped,
        "out_dir_dbfs": out_dir_dbfs,
        "png_path": png_path
    }

# --- Example usage (replace `model` with your fitted PipelineModel)
# res = grouped_feature_importance_ohe(
#     model,                        # fitted PipelineModel
#     model_name="logreg_ohe_v1",
#     out_base_dbfs="dbfs:/reports_ohe",
#     featuresCol=None,             # or "scaled_features" if LR consumes that
#     id_ohe_col="ID_ohe",
#     concelho_ohe_col="CONCELHO_ohe",
#     sort_by="l2"
# )
# display(pd.read_csv("/dbfs" + res["out_dir_dbfs"][len("dbfs:"):] + "/importance_grouped.csv"))


In [0]:
# Cell 4 — execute grouped importance and inspect outputs

# 1) Run (assumes `model` is your fitted PipelineModel from the OHE pipeline)
res = grouped_feature_importance_ohe(
    pipeline_model= model_w,
    model_name="logreg_ohe_v1",
    out_base_dbfs="dbfs:/reports_ohe",
    featuresCol=None,            # or "scaled_features" if LR consumes a scaled vector
    id_ohe_col="ID_ohe",
    concelho_ohe_col="CONCELHO_ohe",
    sort_by="l2"                 # "l2" | "l1" | "rms_coef" | "mean_abs"
)

# 2) Load saved CSVs back for quick viewing
run_dir_driver = "/dbfs" + res["out_dir_dbfs"][len("dbfs:"):]  # convert DBFS URI → driver path
grouped_csv = os.path.join(run_dir_driver, "importance_grouped.csv")
all_csv     = os.path.join(run_dir_driver, "importance_all.csv")

import pandas as pd
grouped_df = pd.read_csv(grouped_csv)
all_df     = pd.read_csv(all_csv)

# Sorted group table (top → bottom)
display(grouped_df.sort_values("l2", ascending=False))

# 3) Show the saved group bar plot
from PIL import Image
from IPython.display import display as nb_display

png_path = os.path.join(run_dir_driver, "importance_groups_l2.png")
nb_display(Image.open(png_path))
print("Plot:", png_path)


In [0]:
# Expand `num_scaled` into original numeric feature names and plot their importances

import builtins, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.feature import VectorAssembler, StandardScalerModel, OneHotEncoderModel, StringIndexerModel

# --- helpers --------------------------------------------------------------

def _find_lr_and_features_col(pipeline: PipelineModel):
    lr = None
    for st in reversed(pipeline.stages):
        if isinstance(st, LogisticRegressionModel):
            lr = st; break
    if lr is None:
        raise ValueError("No LogisticRegressionModel found in pipeline.")
    return lr, lr.getFeaturesCol()

def _vector_dims(pipeline: PipelineModel):
    """Return: vec_dims (VA output -> dim), scaler_in (scaler out -> its input col)"""
    vec_dims, scaler_in = {}, {}
    for st in pipeline.stages:
        if isinstance(st, VectorAssembler):
            vec_dims[st.getOutputCol()] = len(st.getInputCols())
        elif isinstance(st, StandardScalerModel):
            scaler_in[st.getOutputCol()] = st.getInputCol()
    return vec_dims, scaler_in

def _ohe_meta(pipeline: PipelineModel):
    """Return OHE output cols, sizes, dropLast (or None if no OHE)."""
    for st in pipeline.stages:
        if isinstance(st, OneHotEncoderModel):
            outs = st.getOutputCols() if hasattr(st, "getOutputCols") else [st.getOutputCol()]
            sizes = list(st.categorySizes) if hasattr(st, "categorySizes") else [st.getCategorySizes()]
            return outs, sizes, bool(st.getDropLast())
    return [], [], True

def _resolve_numeric_names_for_scaled(pipeline: PipelineModel, scaled_col="num_scaled"):
    """Find the StandardScaler that outputs `scaled_col`, then its upstream VectorAssembler input names."""
    va_inputs = None
    upstream_col = None
    for st in pipeline.stages:
        if isinstance(st, StandardScalerModel) and st.getOutputCol() == scaled_col:
            upstream_col = st.getInputCol()
            break
    if not upstream_col:
        return None
    for st in pipeline.stages:
        if isinstance(st, VectorAssembler) and st.getOutputCol() == upstream_col:
            va_inputs = list(st.getInputCols())
            break
    return va_inputs  # list of original numeric feature names (order matches vector order)

def _build_feature_names_and_groups_with_override(
    pipeline: PipelineModel,
    featuresCol: str | None,
    id_ohe_col="ID_ohe",
    concelho_ohe_col="CONCELHO_ohe",
    vector_name_overrides: dict[str, list[str]] | None = None
):
    lr, lr_feats = _find_lr_and_features_col(pipeline)
    feats_col = featuresCol or lr_feats

    # final assembler that produces feats_col
    final_va = None
    for st in pipeline.stages:
        if isinstance(st, VectorAssembler) and st.getOutputCol() == feats_col:
            final_va = st; break
    if final_va is None:
        raise ValueError(f"Could not find VectorAssembler that outputs '{feats_col}'.")

    inputs = list(final_va.getInputCols())
    vec_dims, scaler_in = _vector_dims(pipeline)
    ohe_outs, ohe_sizes, dropLast = _ohe_meta(pipeline)
    vector_name_overrides = vector_name_overrides or {}

    def dim_for(colname: str) -> int:
        if colname in vec_dims:
            return int(vec_dims[colname])
        if colname in scaler_in:
            up = scaler_in[colname]
            return int(vec_dims.get(up, 1))
        return 1

    def ohe_dim_for(out_col: str) -> int:
        if out_col in ohe_outs:
            i = ohe_outs.index(out_col)
            base = int(ohe_sizes[i])
        else:
            base = int(ohe_sizes[0]) if ohe_sizes else 0
        return base - (1 if dropLast else 0)

    names, groups = [], []
    for c in inputs:
        # OHE expansion by size, names added elsewhere (we're focusing on numeric now)
        if c in (id_ohe_col, concelho_ohe_col):
            k = ohe_dim_for(c)
            # pad k dummy slots with recognizable tokens; group by the field
            grp = "ID" if c == id_ohe_col else "CONCELHO"
            names.extend([f"{grp}__ohe[{i}]" for i in range(builtins.max(k, 0))])
            groups.extend([grp] * builtins.max(k, 0))
        else:
            if c in vector_name_overrides:
                override = vector_name_overrides[c]
                names.extend(override)
                groups.extend([c] * len(override))
            else:
                d = dim_for(c)
                if d == 1:
                    names.append(c); groups.append(c)
                else:
                    names.extend([f"{c}[{i}]" for i in range(d)])
                    groups.extend([c] * d)
    return names, groups

# --- do the work ----------------------------------------------------------

# Your fitted PipelineModel (the OHE + scaler + assembler + LR one)
pm = model_w               # <-- change if your variable is different
features_col_for_lr = None # or "scaled_features" if LR uses that name

# 1) get original numeric names (order inside num_scaled)
num_names = _resolve_numeric_names_for_scaled(pm, scaled_col="num_scaled")
if not num_names:
    raise ValueError("Couldn’t resolve the original numeric columns for 'num_scaled'.")

# 2) rebuild full feature name list, but override 'num_scaled' dims with original names
feature_names, groups = _build_feature_names_and_groups_with_override(
    pm,
    featuresCol=features_col_for_lr,
    id_ohe_col="ID_ohe",
    concelho_ohe_col="CONCELHO_ohe",
    vector_name_overrides={"num_scaled": num_names}
)

# 3) coefficients aligned to those names
lr, _ = _find_lr_and_features_col(pm)
coefs = np.array(lr.coefficients.toArray())

if len(feature_names) != len(coefs):
    raise ValueError(f"Length mismatch: names={len(feature_names)} vs coefs={len(coefs)}")

all_df = pd.DataFrame({
    "feature": feature_names,
    "group": groups,
    "coef": coefs,
    "abs_coef": np.abs(coefs)
})

# 4) extract just the numeric block and save/plot
num_df = all_df[all_df["group"] == "num_scaled"].copy()

ts = time.strftime("%Y%m%d_%H%M%S")
base_dbfs = "dbfs:/reports_ohe"
out_dir_dbfs = f"{base_dbfs}/logreg_ohe_v1/{ts}_num_scaled"
out_dir_driver = "/dbfs" + out_dir_dbfs[len("dbfs:"):]
os.makedirs(out_dir_driver, exist_ok=True)

csv_path = os.path.join(out_dir_driver, "num_scaled_importance.csv")
num_df.to_csv(csv_path, index=False)

# bar chart (top K)
K = 25
top = num_df.sort_values("abs_coef", ascending=False).head(K).iloc[::-1]
plt.figure(figsize=(10, builtins.max(4.0, 0.35*len(top))))
plt.barh(top["feature"], top["abs_coef"])
plt.title(f"Numeric features — top {len(top)} | |coef|")
plt.xlabel("|coef|"); plt.tight_layout()
png_path = os.path.join(out_dir_driver, "num_scaled_importance_top.png")
plt.savefig(png_path, dpi=160, bbox_inches="tight"); plt.close()

print("Saved:", csv_path)
print("Saved:", png_path)


In [0]:
png_path = os.path.join(run_dir_driver, "importance_groups_l2.png")
nb_display(Image.open("/dbfs/reports_ohe/logreg_ohe_v1/20250924_112035_num_scaled/num_scaled_importance_top.png"))
print("Plot:", "/dbfs/reports_ohe/logreg_ohe_v1/20250924_112035_num_scaled/num_scaled_importance_top.png")

Hyperparameter Tuning

In [0]:
# ---- safety: disable MLflow pyspark autolog in this cell ----
import mlflow
try:
    mlflow.pyspark.ml.autolog(disable=True)
    mlflow.autolog(disable=True)  # belt & suspenders
except Exception as e:
    print("Autolog disable note:", e)

from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression, LogisticRegressionModel
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# --- config ---
label_col = "has_falha"
features_col = "features"
aupr_evaluator = BinaryClassificationEvaluator(
    labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderPR"
)

# --- class weights on 'train' ---
counts = {r[label_col]: r['count'] for r in train.groupBy(label_col).count().collect()}
pos_n = counts.get(1, 0); neg_n = counts.get(0, 0)
assert pos_n > 0 and neg_n > 0, "Both classes must exist in train."
pos_w = float(neg_n) / float(pos_n)

train_w = train.withColumn(
    "class_weight",
    F.when(F.col(label_col) == 1, F.lit(pos_w)).otherwise(F.lit(1.0))
)

display(
    spark.range(1).select(
        F.lit(neg_n).alias("negatives"),
        F.lit(pos_n).alias("positives"),
        F.lit(pos_w).alias("pos_weight(neg/pos)"),
        F.round(F.lit(pos_w), 3).alias("pos_weight_round")
    )
)

# --- helpers ---
def pipelinemodel_to_estimator(pipe_or_model):
    if isinstance(pipe_or_model, Pipeline):
        return pipe_or_model
    if isinstance(pipe_or_model, PipelineModel):
        new_stages = []
        for s in pipe_or_model.stages:
            if isinstance(s, LogisticRegressionModel):
                new_stages.append(LogisticRegression())
            else:
                new_stages.append(s)
        return Pipeline(stages=new_stages)
    raise TypeError("Expected Pipeline or PipelineModel.")

def get_lr_stage(pipe_estimator: Pipeline) -> LogisticRegression:
    for s in pipe_estimator.getStages():
        if isinstance(s, LogisticRegression):
            return s
    raise ValueError("No LogisticRegression stage found.")

def build_param_grid(lr_est: LogisticRegression):
    # Tune only reg/elastic here (threshold handled later to avoid evaluator concurrency)
    return (ParamGridBuilder()
            .addGrid(lr_est.regParam, [1e-4, 1e-3, 1e-2, 1e-1])
            .addGrid(lr_est.elasticNetParam, [0.0, 0.5, 1.0])
            .build())

def tune_pipeline_aupr(pipeline_in, name: str):
    pipe_est = pipelinemodel_to_estimator(pipeline_in)
    lr = get_lr_stage(pipe_est)
    lr.setLabelCol(label_col).setFeaturesCol(features_col).setWeightCol("class_weight")

    param_grid = build_param_grid(lr)

    cv = CrossValidator(
        estimator=pipe_est,
        estimatorParamMaps=param_grid,
        evaluator=aupr_evaluator,  # AUPR is stable for imbalance & doesn't depend on threshold
        numFolds=3,
        parallelism=1,             # <-- avoid concurrency bug
        seed=42
    )

    cv_model = cv.fit(train_w)

    # summarize trials
    rows = []
    for pm, metric in zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics):
        rows.append((float(pm[lr.regParam]), float(pm[lr.elasticNetParam]), float(metric)))
    res_df = spark.createDataFrame(rows, ["regParam","elasticNetParam","cv_AUPR"])\
                  .orderBy(F.desc("cv_AUPR"))

    print(f"=== {name}: top hyperparams by CV areaUnderPR ===")
    display(res_df.limit(15))

    best_lr_model = next(s for s in cv_model.bestModel.stages if isinstance(s, LogisticRegressionModel))
    best = {
        "regParam": best_lr_model.getRegParam(),
        "elasticNetParam": best_lr_model.getElasticNetParam(),
        "threshold": best_lr_model.getThreshold(),  # will tune next
        "weightCol": best_lr_model.getWeightCol()
    }
    print(f"{name} (AUPR-tuned) best params (pre-threshold): {best}")
    return cv_model, res_df

# ---- run for both pipelines (AUPR tuning) ----
cv_model_w, results_w = tune_pipeline_aupr(model_w, "LR_with_weather")
cv_model_n, results_n = tune_pipeline_aupr(model_n, "LR_no_weather")

# ---------- Phase 2: threshold sweep for F1/Recall ----------
# Use a proper validation df if you have one named `valid`; else take a small holdout from train (quick fallback).
try:
    cal_df = valid
except NameError:
    cal_df = train.sample(False, 0.2, seed=42)

def sweep_threshold(best_cv_model, name: str, thresholds=(0.5,0.45,0.4,0.35,0.3,0.25,0.2)):
    preds = best_cv_model.transform(cal_df).select(
        F.col(label_col).alias("y"),
        vector_to_array(F.col("probability"))[1].alias("p1")  # P(class=1)
    )

    rows = []
    for th in thresholds:
        pr = preds.select(
            "y",
            (F.col("p1") >= F.lit(th)).cast("int").alias("yhat")
        )
        tp = pr.filter((F.col("y")==1) & (F.col("yhat")==1)).count()
        fp = pr.filter((F.col("y")==0) & (F.col("yhat")==1)).count()
        fn = pr.filter((F.col("y")==1) & (F.col("yhat")==0)).count()
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2*precision*recall)/(precision+recall) if (precision+recall) > 0 else 0.0
        rows.append((float(th), float(precision), float(recall), float(f1)))

    out = spark.createDataFrame(rows, ["threshold","precision","recall","f1"]).orderBy(F.desc("f1"))
    print(f"=== {name}: threshold sweep on calibration set ===")
    display(out)
    return out

thr_w = sweep_threshold(cv_model_w.bestModel, "LR_with_weather")
thr_n = sweep_threshold(cv_model_n.bestModel, "LR_no_weather")


In [0]:
# from pyspark.sql import functions as F
# from pyspark.ml.classification import LogisticRegressionModel
# from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# # 1) Inspect top CV results again (to see actual numbers)
# display(results_w.limit(10))
# display(results_n.limit(10))

# # 2) Pick best threshold from the sweep tables (already sorted by f1 desc)
# best_thr_w = thr_w.first()["threshold"]
# best_thr_n = thr_n.first()["threshold"]

# # 3) Apply threshold to best models
# best_model_w = cv_model_w.bestModel
# best_model_n = cv_model_n.bestModel

# lr_w = next(s for s in best_model_w.stages if isinstance(s, LogisticRegressionModel))
# lr_n = next(s for s in best_model_n.stages if isinstance(s, LogisticRegressionModel))

# lr_w.setThreshold(float(best_thr_w))
# lr_n.setThreshold(float(best_thr_n))

# # 4) Choose evaluation df (use your chronological validation set if you have it)
# try:
#     eval_df = valid   # preferred (future slice)
# except NameError:
#     eval_df = cal_df  # fallback

# # 5) Evaluate F1 and Recall
# e_f1  = MulticlassClassificationEvaluator(labelCol="has_falha", metricName="f1")
# e_rec = MulticlassClassificationEvaluator(labelCol="has_falha", metricName="weightedRecall")

# pred_w = best_model_w.transform(eval_df)
# pred_n = best_model_n.transform(eval_df)

# display(
#     spark.createDataFrame([
#         ("with_weather", float(e_f1.evaluate(pred_w)), float(e_rec.evaluate(pred_w)), float(best_thr_w)),
#         ("no_weather" , float(e_f1.evaluate(pred_n)), float(e_rec.evaluate(pred_n)), float(best_thr_n))
#     ], ["model","F1_valid","Recall_valid","chosen_threshold"])
# )

# # 6) (Optional) quick confusion matrices
# def confmat(df, label="has_falha", pred="prediction"):
#     tp = df.filter((F.col(label)==1)&(F.col(pred)==1)).count()
#     fp = df.filter((F.col(label)==0)&(F.col(pred)==1)).count()
#     fn = df.filter((F.col(label)==1)&(F.col(pred)==0)).count()
#     tn = df.filter((F.col(label)==0)&(F.col(pred)==0)).count()
#     return tp, fp, fn, tn

# tp, fp, fn, tn = confmat(pred_w)
# display(spark.createDataFrame([(tp, fp, fn, tn)], ["TP","FP","FN","TN"]).withColumn("model", F.lit("with_weather")))

# tp, fp, fn, tn = confmat(pred_n)
# display(spark.createDataFrame([(tp, fp, fn, tn)], ["TP","FP","FN","TN"]).withColumn("model", F.lit("no_weather")))


In [0]:
# from pyspark.sql import functions as F
# from pyspark.ml.classification import LogisticRegressionModel

# # 1) Show the CV tables (AUPR tuning)
# print("=== CV (AUPR) results — with weather ===")
# display(results_w.orderBy(F.desc("cv_AUPR")).limit(15))

# print("=== CV (AUPR) results — no weather ===")
# display(results_n.orderBy(F.desc("cv_AUPR")).limit(15))

# # 2) Show the threshold sweeps (already sorted by F1 in our code)
# print("=== Threshold sweep — with weather ===")
# display(thr_w)

# print("=== Threshold sweep — no weather ===")
# display(thr_n)

# # 3) Pick best thresholds from the sweep tables
# best_thr_w = thr_w.orderBy(F.desc("f1"), F.desc("recall")).first()["threshold"]
# best_thr_n = thr_n.orderBy(F.desc("f1"), F.desc("recall")).first()["threshold"]

# # 4) Apply thresholds to best models and evaluate on your validation slice (prefer `valid`, fallback was cal_df)
# best_model_w = cv_model_w.bestModel
# best_model_n = cv_model_n.bestModel

# lr_w = next(s for s in best_model_w.stages if isinstance(s, LogisticRegressionModel))
# lr_n = next(s for s in best_model_n.stages if isinstance(s, LogisticRegressionModel))
# lr_w.setThreshold(float(best_thr_w))
# lr_n.setThreshold(float(best_thr_n))

# try:
#     eval_df = valid
#     print("Evaluating on `valid`")
# except NameError:
#     eval_df = cal_df
#     print("Evaluating on `cal_df` (20% sample of `train`)")

# # 5) Compute F1 & Recall
# from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# e_f1  = MulticlassClassificationEvaluator(labelCol="has_falha", metricName="f1")
# e_rec = MulticlassClassificationEvaluator(labelCol="has_falha", metricName="weightedRecall")

# pred_w = best_model_w.transform(eval_df)
# pred_n = best_model_n.transform(eval_df)

# summary_df = spark.createDataFrame([
#     ("with_weather" , float(e_f1.evaluate(pred_w)), float(e_rec.evaluate(pred_w)), float(best_thr_w)),
#     ("no_weather"  , float(e_f1.evaluate(pred_n)), float(e_rec.evaluate(pred_n)), float(best_thr_n)),
# ], ["model","F1_valid","Recall_valid","chosen_threshold"])

# print("=== Final validation metrics ===")
# display(summary_df)

# # 6) Confusion matrices
# def confmat(df, label="has_falha", pred="prediction"):
#     tp = df.filter((F.col(label)==1)&(F.col(pred)==1)).count()
#     fp = df.filter((F.col(label)==0)&(F.col(pred)==1)).count()
#     fn = df.filter((F.col(label)==1)&(F.col(pred)==0)).count()
#     tn = df.filter((F.col(label)==0)&(F.col(pred)==0)).count()
#     return tp, fp, fn, tn

# tp, fp, fn, tn = confmat(pred_w)
# print("=== Confusion matrix — with_weather ===")
# display(spark.createDataFrame([(tp, fp, fn, tn)], ["TP","FP","FN","TN"]))

# tp, fp, fn, tn = confmat(pred_n)
# print("=== Confusion matrix — no_weather ===")
# display(spark.createDataFrame([(tp, fp, fn, tn)], ["TP","FP","FN","TN"]))


In [0]:
# from pyspark.sql import functions as F
# from pyspark.ml.classification import LogisticRegressionModel

# # 0) sanity: make sure your calibration/validation slice has positives
# try:
#     cal_df  # created earlier
# except NameError:
#     cal_df = valid if 'valid' in globals() else train.sample(False, 0.2, seed=42)

# pos_cal = cal_df.filter(F.col("has_falha")==1).count()
# neg_cal = cal_df.filter(F.col("has_falha")==0).count()
# if pos_cal == 0:
#     print("⚠️ Your calibration/validation set has 0 positives — F1/Recall & threshold sweeps are meaningless. Use a future slice that includes positives.")
# display(spark.createDataFrame([(neg_cal, pos_cal)], ["negatives_cal","positives_cal"]))

# # 1) show the actual rows for CV & threshold sweeps
# print("=== CV (AUPR) results — with weather (top 15) ===")
# display(results_w.orderBy(F.desc("cv_AUPR")).limit(15))

# print("=== CV (AUPR) results — no weather (top 15) ===")
# display(results_n.orderBy(F.desc("cv_AUPR")).limit(15))

# print("=== Threshold sweep — with weather (sorted by F1, then Recall) ===")
# display(thr_w.orderBy(F.desc("f1"), F.desc("recall")).limit(15))

# print("=== Threshold sweep — no weather (sorted by F1, then Recall) ===")
# display(thr_n.orderBy(F.desc("f1"), F.desc("recall")).limit(15))

# # 2) pick the best threshold (max F1; tie-break by Recall)
# best_thr_w = thr_w.orderBy(F.desc("f1"), F.desc("recall")).first()["threshold"]
# best_thr_n = thr_n.orderBy(F.desc("f1"), F.desc("recall")).first()["threshold"]

# # 3) grab the AUPR-best regularization settings
# best_cv_w = results_w.orderBy(F.desc("cv_AUPR")).first()
# best_cv_n = results_n.orderBy(F.desc("cv_AUPR")).first()

# # 4) apply chosen thresholds to the best CV models
# best_model_w = cv_model_w.bestModel
# best_model_n = cv_model_n.bestModel
# lr_w = next(s for s in best_model_w.stages if isinstance(s, LogisticRegressionModel))
# lr_n = next(s for s in best_model_n.stages if isinstance(s, LogisticRegressionModel))
# lr_w.setThreshold(float(best_thr_w))
# lr_n.setThreshold(float(best_thr_n))

# # 5) evaluate F1 & Recall on your validation/calibration slice
# from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# e_f1  = MulticlassClassificationEvaluator(labelCol="has_falha", metricName="f1")
# e_rec = MulticlassClassificationEvaluator(labelCol="has_falha", metricName="weightedRecall")

# pred_w = best_model_w.transform(cal_df)
# pred_n = best_model_n.transform(cal_df)

# f1_w, rec_w = float(e_f1.evaluate(pred_w)), float(e_rec.evaluate(pred_w))
# f1_n, rec_n = float(e_f1.evaluate(pred_n)), float(e_rec.evaluate(pred_n))

# # 6) final 2-row summary of the winners (this is the bit you “make something of”)
# final_summary = spark.createDataFrame([
#     ("with_weather",
#      float(best_cv_w['regParam']), float(best_cv_w['elasticNetParam']), float(best_thr_w),
#      f1_w, rec_w),
#     ("no_weather",
#      float(best_cv_n['regParam']), float(best_cv_n['elasticNetParam']), float(best_thr_n),
#      f1_n, rec_n)
# ], ["model","regParam","elasticNetParam","chosen_threshold","F1_valid","Recall_valid"])

# print("=== Final selection summary (pick by F1, or by Recall if that’s your priority) ===")
# display(final_summary)

# # 7) optional: confusion matrices
# def confmat(df, label="has_falha", pred="prediction"):
#     tp = df.filter((F.col(label)==1)&(F.col(pred)==1)).count()
#     fp = df.filter((F.col(label)==0)&(F.col(pred)==1)).count()
#     fn = df.filter((F.col(label)==1)&(F.col(pred)==0)).count()
#     tn = df.filter((F.col(label)==0)&(F.col(pred)==0)).count()
#     return tp, fp, fn, tn

# tp, fp, fn, tn = confmat(pred_w)
# print("=== Confusion matrix — with_weather ===")
# display(spark.createDataFrame([(tp, fp, fn, tn)], ["TP","FP","FN","TN"]))

# tp, fp, fn, tn = confmat(pred_n)
# print("=== Confusion matrix — no_weather ===")
# display(spark.createDataFrame([(tp, fp, fn, tn)], ["TP","FP","FN","TN"]))


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import builtins

# ---- helpers ----
def _confmat(pred_df, label="has_falha", pred="prediction"):
    agg = pred_df.agg(
        F.sum(((F.col(label)==1)&(F.col(pred)==1)).cast("int")).alias("TP"),
        F.sum(((F.col(label)==0)&(F.col(pred)==1)).cast("int")).alias("FP"),
        F.sum(((F.col(label)==1)&(F.col(pred)==0)).cast("int")).alias("FN"),
        F.sum(((F.col(label)==0)&(F.col(pred)==0)).cast("int")).alias("TN"),
    ).collect()[0]
    return int(agg["TP"]), int(agg["FP"]), int(agg["FN"]), int(agg["TN"])

def _best_by(df, cols_desc):
    # cols_desc like ["cv_AUPR"] or ["f1","recall"]
    order = [F.desc(c) for c in cols_desc]
    return df.orderBy(*order).first()

# ---- sanity on calibration set ----
pos_cal = cal_df.filter(F.col("has_falha")==1).count()
neg_cal = cal_df.filter(F.col("has_falha")==0).count()
print(f"[calibration] positives={pos_cal}, negatives={neg_cal}")
if pos_cal == 0:
    print("⚠️ No positives in calibration set — thresholds & F1/Recall are not meaningful. Use a future slice with positives.")

# ---- pick best hyperparams (AUPR) and thresholds (F1) ----
best_cv_w = _best_by(results_w, ["cv_AUPR"])
best_cv_n = _best_by(results_n, ["cv_AUPR"])
best_thr_w = _best_by(thr_w, ["f1","recall"])["threshold"]
best_thr_n = _best_by(thr_n, ["f1","recall"])["threshold"]

# ---- apply thresholds to best models ----
best_model_w = cv_model_w.bestModel
best_model_n = cv_model_n.bestModel
lr_w = next(s for s in best_model_w.stages if isinstance(s, LogisticRegressionModel))
lr_n = next(s for s in best_model_n.stages if isinstance(s, LogisticRegressionModel))
lr_w.setThreshold(float(best_thr_w))
lr_n.setThreshold(float(best_thr_n))

# ---- evaluate on calibration/validation slice ----
e_f1  = MulticlassClassificationEvaluator(labelCol="has_falha", metricName="f1")
e_rec = MulticlassClassificationEvaluator(labelCol="has_falha", metricName="weightedRecall")

pred_w = best_model_w.transform(cal_df)
pred_n = best_model_n.transform(cal_df)

f1_w, rec_w = float(e_f1.evaluate(pred_w)), float(e_rec.evaluate(pred_w))
f1_n, rec_n = float(e_f1.evaluate(pred_n)), float(e_rec.evaluate(pred_n))
tp_w, fp_w, fn_w, tn_w = _confmat(pred_w)
tp_n, fp_n, fn_n, tn_n = _confmat(pred_n)

# ---- concise, human-readable summary (no display() dependency) ----
# deltas using built-in abs (avoid pyspark F.abs)
delta_f1 = builtins.abs(float(f1_w) - float(f1_n))
delta_rec = builtins.abs(float(rec_w) - float(rec_n))

print("\n=== FINAL SELECTION SUMMARY ===")
print(f"[with_weather]  regParam={best_cv_w['regParam']:.3g}  elasticNet={best_cv_w['elasticNetParam']:.3g}  "
      f"threshold={best_thr_w:.3g}  F1={f1_w:.4f}  Recall={rec_w:.4f}  "
      f"TP={tp_w} FP={fp_w} FN={fn_w} TN={tn_w}")

print(f"[no_weather]   regParam={best_cv_n['regParam']:.3g}  elasticNet={best_cv_n['elasticNetParam']:.3g}  "
      f"threshold={best_thr_n:.3g}  F1={f1_n:.4f}  Recall={rec_n:.4f}  "
      f"TP={tp_n} FP={fp_n} FN={tn_n} TN={tn_n}")

winner = "with_weather" if f1_w > f1_n else "no_weather"
print(f"\n-> Winner by F1: {winner} (ΔF1={delta_f1:.4f}, ΔRecall={delta_rec:.4f})")

# ---- optional tiny table fallback (safe even if display() is quirky) ----
try:
    import pandas as pd
    rows = [
        ["with_weather", float(best_cv_w['regParam']), float(best_cv_w['elasticNetParam']), float(best_thr_w),
         f1_w, rec_w, tp_w, fp_w, fn_w, tn_w],
        ["no_weather",  float(best_cv_n['regParam']), float(best_cv_n['elasticNetParam']), float(best_thr_n),
         f1_n, rec_n, tp_n, fp_n, fn_n, tn_n],
    ]
    pd_df = pd.DataFrame(rows, columns=["model","regParam","elasticNet","threshold","F1","Recall","TP","FP","FN","TN"])
    print("\n(as table)")
    print(pd_df.to_string(index=False))
except Exception as e:
    pass


In [0]:
# Use a real chronological validation slice (replace this with your split)
valid = final_df.filter(F.col("DATE") >= F.lit("2023-11-25"))  # example

from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.functions import vector_to_array
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

label_col = "has_falha"
thresholds = [i/100 for i in range(95, 4, -5)]  # 0.95 -> 0.05

def sweep(model, df, name):
    preds = model.transform(df).select(
        F.col(label_col).alias("y"),
        vector_to_array(F.col("probability"))[1].alias("p1")
    ).cache()

    rows=[]
    for th in thresholds:
        agg = preds.agg(
            F.sum(((F.col("y")==1) & (F.col("p1")>=th)).cast("int")).alias("tp"),
            F.sum(((F.col("y")==0) & (F.col("p1")>=th)).cast("int")).alias("fp"),
            F.sum(((F.col("y")==1) & (F.col("p1")< th)).cast("int")).alias("fn"),
            F.sum(((F.col("y")==0) & (F.col("p1")< th)).cast("int")).alias("tn"),
        ).first()
        tp, fp, fn, tn = [int(agg[c]) for c in ("tp","fp","fn","tn")]
        prec = tp/(tp+fp) if (tp+fp) else 0.0
        rec  = tp/(tp+fn) if (tp+fn) else 0.0
        f1   = (2*prec*rec)/(prec+rec) if (prec+rec) else 0.0
        rows.append((th, prec, rec, f1, tp, fp, fn, tn))
    out = spark.createDataFrame(rows, ["threshold","precision","recall","f1","TP","FP","FN","TN"])\
               .orderBy(F.desc("f1"), F.desc("recall"))
    print(f"=== Dense threshold sweep on VALID — {name} ===")
    display(out.limit(20))
    return out

best_w_valid = sweep(cv_model_w.bestModel, valid, "with_weather")
best_n_valid = sweep(cv_model_n.bestModel, valid, "no_weather")


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
import pandas as pd
import builtins  # to avoid PySpark function shadowing of abs/round

label_col = "has_falha"

def show_top(df, title, n=10):
    pdf = df.orderBy(F.desc("f1"), F.desc("recall")).limit(n).toPandas()
    print(f"\n=== {title} (top {n}) ===")
    print(pdf.to_string(index=False))

# 1) Show top thresholds (prints actual rows)
show_top(best_w_valid, "VALID sweep — with_weather", n=15)
show_top(best_n_valid, "VALID sweep — no_weather", n=15)

# 2) Pick best thresholds from the VALID sweeps
thr_w_star = best_w_valid.orderBy(F.desc("f1"), F.desc("recall")).first()["threshold"]
thr_n_star = best_n_valid.orderBy(F.desc("f1"), F.desc("recall")).first()["threshold"]

# 3) Evaluate F1/Recall on VALID at those thresholds (without relying on display)
def evaluate_at_threshold(model, df, thr):
    # Get probabilities
    proba = model.transform(df).select(
        F.col(label_col).cast("int").alias("y"),
        vector_to_array(F.col("probability"))[1].alias("p1")
    )
    # Post-hoc thresholding (keep ints for counting)
    pred = proba.select(
        "y",
        (F.col("p1") >= F.lit(float(thr))).cast("int").alias("yhat")
    )
    # Confusion counts
    agg = pred.agg(
        F.sum(((F.col("y")==1) & (F.col("yhat")==1)).cast("int")).alias("TP"),
        F.sum(((F.col("y")==0) & (F.col("yhat")==1)).cast("int")).alias("FP"),
        F.sum(((F.col("y")==1) & (F.col("yhat")==0)).cast("int")).alias("FN"),
        F.sum(((F.col("y")==0) & (F.col("yhat")==0)).cast("int")).alias("TN"),
    ).first()

    tp, fp, fn, tn = int(agg["TP"]), int(agg["FP"]), int(agg["FN"]), int(agg["TN"])
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0
    return f1, recall, tp, fp, fn, tn

f1_w, rec_w, tp_w, fp_w, fn_w, tn_w = evaluate_at_threshold(cv_model_w.bestModel, valid, thr_w_star)
f1_n, rec_n, tp_n, fp_n, fn_n, tn_n = evaluate_at_threshold(cv_model_n.bestModel, valid, thr_n_star)

print("\n=== FINAL VALIDATION SUMMARY (chosen thresholds from VALID sweep) ===")
print(f"[with_weather]  thr={thr_w_star:.3f}  F1={f1_w:.4f}  Recall={rec_w:.4f}  TP={tp_w} FP={fp_w} FN={fn_w} TN={tn_w}")
print(f"[no_weather]   thr={thr_n_star:.3f}  F1={f1_n:.4f}  Recall={rec_n:.4f}  TP={tp_n} FP={fp_n} FN={fn_n} TN={tn_n}")


# 5) Small table print (also pure text)
summary_pdf = pd.DataFrame([
    ["with_weather", float(thr_w_star), f1_w, rec_w, tp_w, fp_w, fn_w, tn_w],
    ["no_weather",  float(thr_n_star), f1_n, rec_n, tp_n, fp_n, fn_n, tn_n],
], columns=["model","threshold","F1_valid","Recall_valid","TP","FP","FN","TN"])
print("\n(as table)")
print(summary_pdf.to_string(index=False))



In [0]:
from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.functions import vector_to_array

label_col = "has_falha"
chosen_thr = 0.75  # winner from VALID
champ = cv_model_w.bestModel  # with_weather winner

def eval_on(model, df, thr):
    proba = model.transform(df).select(
        F.col(label_col).cast("int").alias("y"),
        vector_to_array(F.col("probability"))[1].alias("p1")
    )
    pred = proba.select("y", (F.col("p1") >= F.lit(float(thr))).cast("int").alias("yhat"))
    agg = pred.agg(
        F.sum(((F.col("y")==1)&(F.col("yhat")==1)).cast("int")).alias("TP"),
        F.sum(((F.col("y")==0)&(F.col("yhat")==1)).cast("int")).alias("FP"),
        F.sum(((F.col("y")==1)&(F.col("yhat")==0)).cast("int")).alias("FN"),
        F.sum(((F.col("y")==0)&(F.col("yhat")==0)).cast("int")).alias("TN"),
    ).first()
    tp, fp, fn, tn = int(agg["TP"]), int(agg["FP"]), int(agg["FN"]), int(agg["TN"])
    prec = tp/(tp+fp) if (tp+fp) else 0.0
    rec  = tp/(tp+fn) if (tp+fn) else 0.0
    f1   = (2*prec*rec)/(prec+rec) if (prec+rec) else 0.0
    return prec, rec, f1, tp, fp, fn, tn

# Evaluate on your untouched test set
prec, rec, f1, tp, fp, fn, tn = eval_on(champ, test, chosen_thr)
print(f"[TEST] with_weather @ thr={chosen_thr:.2f}  F1={f1:.4f}  Recall={rec:.4f}  Precision={prec:.4f}  "
      f"TP={tp} FP={fp} FN={fn} TN={tn}")


[TEST] with_weather @ thr=0.85  F1=0.8889  Recall=0.8406  Precision=0.9432  TP=110499 FP=6655 FN=20956 TN=785869

[TEST] with_weather @ thr=0.80  F1=0.8642  Recall=0.8565  Precision=0.8721  TP=112591 FP=16507 FN=18864 TN=776017

[TEST] with_weather @ thr=0.75  F1=0.8416  Recall=0.8694  Precision=0.8155  TP=114284 FP=25859 FN=17171 TN=766665